# 01. 데이터 적재와 임베딩 — KLUE-MRC → PostgreSQL

진행 순서
1. 데이터셋 : 볼륨(`/cache/hf`)에 없을 때만 허깅페이스에서 받는다
2. 스키마 확인 : `embedding vector(4096)` + `emb128 vector(128)` 인지 확인하고 아니면 바꾼다
3. 테이블 초기화 : `TRUNCATE` 로 비우고 지문·질문 텍스트를 먼저 전부 넣는다 → 이 시점부터 키워드 검색(ts_rank·BM25) 실습 가능
4. 임베딩 : 비어 있는 행만 골라 **배치 단위**로 LM Studio 에 보내고, 배치마다 UPDATE → COMMIT
5. 인덱스 : 임베딩이 다 차면 `emb128` 에 HNSW 를 만든다

한 번에 큰 부하를 주지 않도록 배치 크기와 배치 사이 쉬는 시간을 조절할 수 있다.
중간에 멈춰도(Kernel → Interrupt Kernel) 채운 행은 남는다. 「4. 임베딩 서버 확인」 제목 셀을 클릭하고 **Run → Run Selected Cell and All Below** 하면 비어 있는 행부터 이어서 채운다. (Run All Cells 는 3번에서 테이블을 다시 비우므로 쓰지 않는다)

준비 : LM Studio 에서 Qwen3-Embedding-8B 를 로드하고 서버를 켠다. 포트가 1234 가 아니면 compose 기동 때 `LMSTUDIO_URL` 을 넘긴다.

In [ ]:
# ── 설정 ─────────────────────────────────────────────────────────────
SPLIT      = "validation"   # KLUE-MRC 분할 (validation: 지문 5,309 · 질문 5,841)
BATCH_SIZE = 16             # 한 번에 LM Studio 로 보내는 문장 수
PAUSE_SEC  = 0.0            # 배치 사이 쉬는 시간(초). PC 가 버거우면 0.5~2 로
MAX_EMBED  = None           # 이번 실행에서 테이블당 최대 몇 건 임베딩할지 (None = 전부)

# 질의 쪽 instruction (문서 쪽에는 붙이지 않는다). Qwen3-Embedding 은 영어 지시문을 권장
QUERY_TASK = "Given a Korean question, retrieve the news or Wikipedia passage that answers the question"

In [ ]:
import hashlib
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
import requests
from pgvector.psycopg import register_vector

DSN = os.environ.get("DATABASE_URL", "postgresql://lab:lab@db:5432/lab")
LMSTUDIO_URL = os.environ.get("LMSTUDIO_URL", "http://host.docker.internal:1234/v1").rstrip("/")
HF_HOME = Path(os.environ.get("HF_HOME", "/cache/hf"))

conn = psycopg.connect(DSN)          # 배치마다 직접 COMMIT 한다
register_vector(conn)                # numpy 배열 ↔ vector 타입


def one(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        row = cur.fetchone()
    return row[0] if row else None


print("DB          :", one("SELECT version()").split(",")[0])
print("pgvector    :", one("SELECT extversion FROM pg_extension WHERE extname = 'vector'"))
print("LMSTUDIO_URL:", LMSTUDIO_URL)

## 1. 데이터셋 — 볼륨에 없을 때만 다운로드
허깅페이스가 자동 변환해 둔 parquet 파일을 받는다. 저장 위치는 도커 볼륨 `hfcache` 라서 컨테이너를 지워도 남는다.

In [ ]:
path = HF_HOME / "parquet" / f"mrc-{SPLIT}.parquet"
if path.exists():
    print(f"볼륨에 있음 → 다운로드 생략 : {path} ({path.stat().st_size / 2**20:.1f} MB)")
else:
    url = f"https://huggingface.co/api/datasets/klue/klue/parquet/mrc/{SPLIT}/0.parquet"
    print("다운로드 :", url)
    path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    tmp = path.with_suffix(".part")              # 받는 도중 끊겨도 반쪽 파일이 남지 않게
    tmp.write_bytes(r.content)
    tmp.rename(path)
    print(f"저장 : {path} ({path.stat().st_size / 2**20:.1f} MB)")

raw = pd.read_parquet(path)
# 원본은 질문 단위라 같은 지문이 여러 번 나온다 → 지문 내용의 md5 로 중복 제거
raw["doc_key"] = raw["context"].map(lambda s: hashlib.md5(s.encode()).hexdigest())
docs = (raw.drop_duplicates("doc_key")
           .loc[:, ["doc_key", "title", "context", "news_category", "source"]]
           .rename(columns={"context": "content", "news_category": "category"})
           .reset_index(drop=True))
docs["category"] = docs["category"].replace({"": None})
qs = raw.loc[:, ["guid", "doc_key", "question", "question_type", "is_impossible"]].copy()
qs["answer"] = raw["answers"].map(lambda a: a["text"][0] if len(a["text"]) else None)
print(f"질문 {len(raw):,}개 → 지문 {len(docs):,}개 / 질문 {len(qs):,}개")

## 2. 스키마 확인 — 원본 4096 + 128차원 생성 컬럼
- `embedding vector(4096)` : Qwen3-Embedding-8B 원본. HNSW 인덱스는 2,000차원까지라 인덱스 없이 보관 (재정렬·차원 비교용)
- `emb128 vector(128)` : `l2_normalize(subvector(embedding, 1, 128))` 생성 컬럼. 후보 추리기용 HNSW 는 여기에

예전 스키마(MiniLM 384차원)로 만든 DB 볼륨이면 여기서 컬럼을 바꾼다. 어차피 3번에서 테이블을 비우므로 잃는 것은 없다.

In [ ]:
def coltype(table, col):
    return one("""SELECT format_type(atttypid, atttypmod) FROM pg_attribute
                  WHERE attrelid = %s::regclass AND attname = %s AND NOT attisdropped""", (table, col))


with conn.cursor() as cur:
    for t in ["docs", "questions"]:
        before = (coltype(t, "embedding"), coltype(t, "emb128"))
        if before == ("vector(4096)", "vector(128)"):
            print(f"{t:9s} : 이미 새 스키마 {before}")
            continue
        cur.execute("DROP INDEX IF EXISTS docs_embedding_hnsw")
        cur.execute("DROP INDEX IF EXISTS docs_emb128_hnsw")
        cur.execute(f"ALTER TABLE {t} DROP COLUMN IF EXISTS emb128")
        cur.execute(f"ALTER TABLE {t} DROP COLUMN IF EXISTS embedding")
        cur.execute(f"ALTER TABLE {t} ADD COLUMN embedding vector(4096)")
        cur.execute(f"""ALTER TABLE {t} ADD COLUMN emb128 vector(128)
                        GENERATED ALWAYS AS (l2_normalize(subvector(embedding, 1, 128))::vector(128)) STORED""")
        print(f"{t:9s} : {before} → {(coltype(t, 'embedding'), coltype(t, 'emb128'))}")
conn.commit()

## 3. 테이블 초기화 — TRUNCATE 후 텍스트 먼저 INSERT
- `TRUNCATE … RESTART IDENTITY` : 행을 한 번에 비우고 id 를 1부터 다시 매긴다 → 몇 번을 다시 적재해도 같은 지문은 같은 id
- `CASCADE` : docs 를 참조하는 테이블(4-2 의 `doc_chunks`)도 함께 비운다
- `tsv` 는 생성 컬럼이라 COPY 하는 동안 형태소 분석까지 끝난다
- TRUNCATE 는 행 삭제 트리거를 부르지 않는다 → BM25 역색인(sql/20~23)이 있으면 `bm25_rebuild()` 로 다시 맞춘다

In [ ]:
t0 = time.time()
with conn.cursor() as cur:
    cur.execute("DROP INDEX IF EXISTS docs_emb128_hnsw")        # 임베딩을 채우는 동안에는 인덱스를 두지 않는다
    cur.execute("TRUNCATE docs, questions RESTART IDENTITY CASCADE")
    with cur.copy("COPY docs (doc_key, title, content, category, source) FROM STDIN") as cp:
        for d in docs.itertuples(index=False):
            cp.write_row((d.doc_key, d.title, d.content, d.category, d.source))
    cur.execute("SELECT doc_key, id FROM docs")
    ids = dict(cur.fetchall())
    with cur.copy("COPY questions (guid, doc_id, question, answer, question_type, is_impossible) FROM STDIN") as cp:
        for q in qs.itertuples(index=False):
            cp.write_row((q.guid, ids[q.doc_key], q.question, q.answer, int(q.question_type), bool(q.is_impossible)))
conn.commit()
print(f"텍스트 적재 {time.time() - t0:.1f}s : docs {one('SELECT count(*) FROM docs'):,} · questions {one('SELECT count(*) FROM questions'):,}")

if one("SELECT to_regproc('bm25_rebuild') IS NOT NULL"):
    t0 = time.time()
    with conn.cursor() as cur:
        cur.execute("SELECT bm25_rebuild()")
    conn.commit()
    print(f"BM25 역색인 재구축 {time.time() - t0:.1f}s")
with conn.cursor() as cur:
    cur.execute("ANALYZE docs")
    cur.execute("ANALYZE questions")
conn.commit()
print("✔ 텍스트 적재 완료 — 임베딩은 이 노트북에서 계속된다. 키워드 검색 실습은 다른 노트북에서 바로 시작해도 된다")

## 4. 임베딩 서버 확인
`/v1/models` 에서 Qwen3 임베딩 모델을 찾는다. 모델 이름은 설치 환경마다 조금 다르다 (예: `text-embedding-qwen3-embedding-8b`).

In [ ]:
models = [m["id"] for m in requests.get(f"{LMSTUDIO_URL}/models", timeout=10).json()["data"]]
cands = [m for m in models if "qwen3-embedding" in m.lower()]
MODEL = os.environ.get("EMBED_MODEL") or (cands[0] if cands else None)
assert MODEL, f"LM Studio 에 Qwen3-Embedding 모델이 로드되어 있지 않다. 지금 목록: {models}"


def embed(texts, retries=4):
    """LM Studio(OpenAI 호환 /v1/embeddings)로 여러 문장을 한 번에 임베딩. 실패하면 잠시 쉬고 다시 시도."""
    for attempt in range(retries):
        try:
            r = requests.post(f"{LMSTUDIO_URL}/embeddings", json={"model": MODEL, "input": texts}, timeout=600)
            r.raise_for_status()
            data = r.json()["data"]
            data = sorted(data, key=lambda d: d.get("index", 0))
            return [np.asarray(d["embedding"], dtype=np.float32) for d in data]
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 5 * 2 ** attempt
            print(f"  ! 임베딩 요청 실패({type(e).__name__}: {e}) → {wait}s 뒤 재시도")
            time.sleep(wait)


v = embed(["반응기 온도가 계속 오르는 원인"])[0]
print(f"모델 {MODEL} : {v.size}차원, 길이 {np.linalg.norm(v):.4f}, 앞 128차원 길이 {np.linalg.norm(v[:128]):.4f}")
assert v.size == 4096, "4096차원이 아니다. 모델을 확인할 것"

## 5. 배치 임베딩 — 비어 있는 행만, 배치마다 UPDATE → COMMIT
1. `embedding IS NULL` 인 행을 id 순으로 `BATCH_SIZE` 개 읽는다
2. 그 텍스트를 한 번의 요청으로 임베딩한다
3. 받은 벡터로 UPDATE 하고 COMMIT 한다 → `emb128` 은 생성 컬럼이라 DB 가 자동으로 채운다
4. `PAUSE_SEC` 만큼 쉬고 다음 배치

배치마다 커밋하므로 도중에 멈춰도 그때까지 채운 행은 남는다. 이어서 하려면 「4. 임베딩 서버 확인」 제목 셀을 클릭하고 Run → Run Selected Cell and All Below.

In [ ]:
def embed_pending(table, text_sql, prefix="", batch=BATCH_SIZE, limit=MAX_EMBED, report_every=20):
    remaining = one(f"SELECT count(*) FROM {table} WHERE embedding IS NULL")
    todo = remaining if limit is None else min(remaining, limit)
    print(f"[{table}] 비어 있는 행 {remaining:,} → 이번에 {todo:,}건 (배치 {batch}, 쉬는 시간 {PAUSE_SEC}s)")
    done, nb, t0 = 0, 0, time.time()
    while done < todo:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, {text_sql} FROM {table} WHERE embedding IS NULL ORDER BY id LIMIT %s",
                        (min(batch, todo - done),))
            rows = cur.fetchall()
        if not rows:
            break
        vecs = embed([prefix + text for _, text in rows])
        with conn.cursor() as cur:
            cur.executemany(f"UPDATE {table} SET embedding = %s WHERE id = %s",
                            [(vec, id_) for (id_, _), vec in zip(rows, vecs)])
        conn.commit()
        done += len(rows)
        nb += 1
        if nb % report_every == 0 or done >= todo:
            el = time.time() - t0
            rate = done / el
            print(f"  {done:6,}/{todo:,}  {rate:5.1f}건/s  경과 {el / 60:5.1f}분  남은 시간 약 {(todo - done) / rate / 60:5.1f}분")
        if PAUSE_SEC:
            time.sleep(PAUSE_SEC)
    left = one(f"SELECT count(*) FROM {table} WHERE embedding IS NULL")
    print(f"[{table}] {done:,}건 완료 ({time.time() - t0:.0f}s). 남은 행 {left:,}")
    return left


# 문서 쪽 : 제목 + 본문, instruction 없음
left_docs = embed_pending("docs", "coalesce(title, '') || E'\\n' || content")

In [ ]:
# 질의 쪽 : instruction 을 붙인다 (Qwen3-Embedding 모델카드 형식 'Instruct: {task}\nQuery:{query}')
left_qs = embed_pending("questions", "question", prefix=f"Instruct: {QUERY_TASK}\nQuery:")

## 6. 인덱스와 확인
임베딩이 다 차면 `emb128` 에 HNSW(코사인) 인덱스를 만든다. 원본 4096 에는 인덱스를 만들 수 없다(2,000차원 상한).

In [ ]:
if one("SELECT count(*) FROM docs WHERE embedding IS NULL") == 0:
    t0 = time.time()
    with conn.cursor() as cur:
        cur.execute("CREATE INDEX IF NOT EXISTS docs_emb128_hnsw ON docs USING hnsw (emb128 vector_cosine_ops)")
        cur.execute("ANALYZE docs")
    conn.commit()
    print(f"docs_emb128_hnsw 생성 {time.time() - t0:.1f}s")
else:
    print("docs 임베딩이 아직 남아 있다 → 5번 셀을 다시 실행한 뒤 인덱스를 만든다")

with conn.cursor() as cur:
    cur.execute("""
      SELECT t, count(*) AS 행, count(embedding) AS 임베딩,
             round(avg(vector_norm(embedding))::numeric, 4)                     AS 원본_길이,
             round(avg(vector_norm(subvector(embedding, 1, 128)))::numeric, 4)  AS 앞128_길이,
             round(avg(vector_norm(emb128))::numeric, 4)                        AS emb128_길이
      FROM (SELECT 'docs' t, embedding, emb128 FROM docs
            UNION ALL SELECT 'questions', embedding, emb128 FROM questions) x
      GROUP BY t ORDER BY t""")
    cols = [c.name for c in cur.description]
    print(pd.DataFrame(cur.fetchall(), columns=cols).to_string(index=False))
    cur.execute("""SELECT pg_size_pretty(pg_total_relation_size('docs')),
                          coalesce(pg_size_pretty(pg_relation_size(to_regclass('docs_emb128_hnsw'))), '인덱스 없음')""")
    print("docs 전체 크기 / emb128 HNSW 크기 :", cur.fetchone())

### 맛보기 — 같은 질문을 원본 4096 과 emb128 로 검색
정답 지문(★)이 몇 위에 오는지, 두 방식의 상위 5개가 얼마나 겹치는지 본다. 차원별 정밀 비교는 뒤 실습에서 한다.

In [ ]:
QID = 1
with conn.cursor() as cur:
    cur.execute("SELECT question, doc_id FROM questions WHERE id = %s", (QID,))
    question, gold = cur.fetchone()
    print("질문 :", question, f"(정답 지문 id {gold})\n")
    for name, sql in [
        ("원본 4096 전수 비교", "SELECT d.id, d.title, 1 - (d.embedding <=> q.embedding) FROM docs d, questions q "
                                "WHERE q.id = %s ORDER BY d.embedding <=> q.embedding LIMIT 5"),
        ("emb128 (HNSW)",       "SELECT d.id, d.title, 1 - (d.emb128 <=> q.emb128) FROM docs d, questions q "
                                "WHERE q.id = %s ORDER BY d.emb128 <=> q.emb128 LIMIT 5"),
    ]:
        cur.execute(sql, (QID,))
        print(f"[{name}]")
        for id_, title, sim in cur.fetchall():
            print(f"  {'★' if id_ == gold else ' '} {sim:.4f}  {id_:5d}  {title}")
        print()